# STIN MARL Environment Test

本 Notebook 测试 STIN（Satellite-Terrestrial Integrated Network）多智能体强化学习环境的完整功能。

## 关键修复说明

**问题**：`KeyError: 'cpuMaxFrequency not a valid key for sat_args'`

**原因**：`@default_args` 装饰器被错误地放在了 `__init__` 方法上，而不是 `setup_xxx` 方法上。

**正确用法**：
- ❌ 错误：`@default_args(...) def __init__(self, ...)`
- ✅ 正确：`@default_args(...) def setup_cpu_power_sink(self, ...)`

BSK-RL 框架通过 `collect_default_args()` 函数从所有 `setup_xxx` 方法的 `defaults` 属性中收集参数。

In [3]:
from bsk_rl import ConstellationTasking
from bsk_rl.data import STINTaskReward
from bsk_rl.scene import CityTaskScenario
from bsk_rl.sats import ComputationSatellite
from bsk_rl.comm import LOSMultiCommunication
from bsk_rl.utils.orbital import walker_delta_args

In [4]:
def create_stin_environment():
    """创建完整的 STIN MARL 环境。"""
    '''
    constellation_args = walker_delta_args(
        n_planes=6,           # 铱星有 6 个轨道面
        n_sats_per_plane=11,  # 每个面 11 颗工作卫星 (总计 66 颗)
        altitude=781,         # [km] 标准高度约为 781 km
        inc=86.4,             # [deg] 轨道倾角 (近极地轨道)
        clusterspacing=2,     # [F参数] 铱星的相位因子是 2 (Walker 66/6/2)
    )
    '''
    # Walker Delta 星座配置（6 颗卫星，800km 高度，60° 倾角）
    constellation_args = walker_delta_args(
        n_planes=2,           # 2 个轨道面
        altitude=800,         # [km] 轨道高度
        inc=60,               # [deg] 轨道倾角
        clusterspacing=5,     # [deg] 卫星间相位差
    )
    
    # 计算任务场景（基于城市分布）
    task_scenario = CityTaskScenario(
        n_tasks=20,                          # 20 个任务
        n_select_from=100,                   # 从前 100 大城市中选择
        data_size_range=(1e6, 10e6),         # 1-10 Mb
        workload_range=(100, 1000),          # 100-1000 cycles/bit
        max_delay_range=(5.0, 20.0),         # 5-20 秒时延约束
        task_arrival_rate=0.1,               # 0.1 tasks/s 到达率
    )
    
    # 奖励函数配置
    reward_system = STINTaskReward(
        base_reward=1.0,        # 基础完成奖励
        timeout_penalty=2.0,    # 超时惩罚
        delay_weight=0.5,       # 时延权重
        energy_weight=0.1,      # 能耗权重
        balance_weight=0.05,    # 负载均衡权重
    )
    
    # 创建环境
    env = ConstellationTasking(
        satellites=[
            ComputationSatellite(
                name=f"sat-{i}",
                sat_args={
                    "cpuMaxFrequency": 2.0e9,     # 2 GHz
                    "cpuMinFrequency": 0.5e9,     # 0.5 GHz
                    "cpuWorkload": 500.0,         # 500 cycles/bit
                    "batteryStorageCapacity": 80.0 * 3600,  # 80 Wh
                    "taskMinimumElevation": 10.0 * 3.14159 / 180,  # 10°
                }
            )
            for i in range(6)
        ],
        scenario=task_scenario,
        rewarder=reward_system,
        communicator=LOSMultiCommunication(),
        sat_arg_randomizer=constellation_args,
        max_step_duration=60.0,  # 60 秒 per step
        time_limit=3600.0,       # 1 小时 episode
        log_level="INFO",
    )
    
    return env

In [5]:
# ⚠️ 需要重新加载模块（修复了 action_spec 的定义）
import sys
modules_to_delete = [mod for mod in sys.modules.keys() if mod.startswith('bsk_rl')]
for mod_name in modules_to_delete:
    del sys.modules[mod_name]
print(f"已清除 {len(modules_to_delete)} 个缓存模块")

# 重新导入
from bsk_rl import ConstellationTasking
from bsk_rl.data import STINTaskReward
from bsk_rl.scene import CityTaskScenario
from bsk_rl.sats import ComputationSatellite
from bsk_rl.comm import LOSMultiCommunication
from bsk_rl.utils.orbital import walker_delta_args

# 测试环境创建
print("\n开始创建环境...")
env = create_stin_environment()
print("✓ 环境创建成功！")

print("\n重置环境...")
env.reset()
print("✓ 环境重置成功！")

print("\n观测空间:")
env.observation_spaces


d:\miniconda\envs\Basilisk\Lib\site-packages\gymnasium\envs\registration.py:636: UserWarning: WARN: Overriding environment GeneralSatelliteTasking-v1 already in registry.
  logger.warn(f"Overriding environment {new_spec.id} already in registry.")
d:\miniconda\envs\Basilisk\Lib\site-packages\gymnasium\envs\registration.py:636: UserWarning: WARN: Overriding environment SatelliteTasking-v1 already in registry.
  logger.warn(f"Overriding environment {new_spec.id} already in registry.")
d:\miniconda\envs\Basilisk\Lib\site-packages\gymnasium\envs\registration.py:636: UserWarning: WARN: Overriding environment ConstellationTasking-v1 already in registry.
  logger.warn(f"Overriding environment {new_spec.id} already in registry.")
2025-12-17 14:58:50,469 gym                            INFO       Resetting environment with seed=2399955104
2025-12-17 14:58:50,471 scene.stin_scenario            INFO       Generating 20 computation tasks


已清除 52 个缓存模块

开始创建环境...
✓ 环境创建成功！

重置环境...


2025-12-17 14:58:50,909 sats.satellite.sat-0           INFO       <0.00> sat-0: Finding opportunity windows from 0.00 to 3600.00 seconds
2025-12-17 14:58:50,921 sats.satellite.sat-1           INFO       <0.00> sat-1: Finding opportunity windows from 0.00 to 3600.00 seconds
2025-12-17 14:58:50,935 sats.satellite.sat-2           INFO       <0.00> sat-2: Finding opportunity windows from 0.00 to 3600.00 seconds
2025-12-17 14:58:50,945 sats.satellite.sat-3           INFO       <0.00> sat-3: Finding opportunity windows from 0.00 to 3600.00 seconds
2025-12-17 14:58:50,956 sats.satellite.sat-4           INFO       <0.00> sat-4: Finding opportunity windows from 0.00 to 3600.00 seconds
2025-12-17 14:58:50,965 sats.satellite.sat-5           INFO       <0.00> sat-5: Finding opportunity windows from 0.00 to 3600.00 seconds
2025-12-17 14:58:51,154 gym                            INFO       <0.00> Environment reset


✓ 环境重置成功！

观测空间:


{'sat-0': Box(-1e+16, 1e+16, (39,), float64),
 'sat-1': Box(-1e+16, 1e+16, (39,), float64),
 'sat-2': Box(-1e+16, 1e+16, (39,), float64),
 'sat-3': Box(-1e+16, 1e+16, (39,), float64),
 'sat-4': Box(-1e+16, 1e+16, (39,), float64),
 'sat-5': Box(-1e+16, 1e+16, (39,), float64)}

In [6]:
env.action_spaces

{'sat-0': Box(0.0, 1.0, (10,), float64),
 'sat-1': Box(0.0, 1.0, (10,), float64),
 'sat-2': Box(0.0, 1.0, (10,), float64),
 'sat-3': Box(0.0, 1.0, (10,), float64),
 'sat-4': Box(0.0, 1.0, (10,), float64),
 'sat-5': Box(0.0, 1.0, (10,), float64)}

In [7]:
def example_training_loop():
    """示例训练循环（使用随机动作）- PettingZoo 多智能体版本。"""
    
    env = create_stin_environment()
    
    # 重置环境
    obs = env.reset()
    
    done = False
    total_reward = 0.0
    step_count = 0
    
    print(f"环境已重置，智能体列表: {list(env.action_spaces.keys())}")
    
    while not done and step_count < 100:  # 限制最大步数以防无限循环
        # PettingZoo 环境使用 action_spaces (字典)，为每个智能体采样动作
        actions = {
            agent: env.action_spaces[agent].sample() 
            for agent in env.action_spaces.keys()
        }
        
        # 执行动作
        obs, rewards, terminated, truncated, info = env.step(actions)
        
        # 累计奖励
        step_reward = sum(rewards.values())
        total_reward += step_reward
        step_count += 1
        
        # 打印步骤信息（每10步打印一次以避免刷屏）
        if step_count % 10 == 0 or step_count <= 3:
            print(f"Step {step_count}: Reward = {step_reward:.3f}, "
                  f"Cumulative = {total_reward:.3f}")
        
        # 检查是否结束
        done = any(terminated.values()) or any(truncated.values())
    
    print(f"\n{'='*60}")
    print(f"Episode finished after {step_count} steps")
    print(f"Total reward: {total_reward:.3f}")
    print(f"Average reward per step: {total_reward/step_count:.3f}")
    print(f"{'='*60}")
    
    env.close()


In [8]:
# 重新加载模块以获取最新修复
import sys
modules_to_delete = [mod for mod in sys.modules.keys() if mod.startswith('bsk_rl')]
for mod_name in modules_to_delete:
    del sys.modules[mod_name]
print(f"已清除 {len(modules_to_delete)} 个缓存模块")

# 重新导入
from bsk_rl import ConstellationTasking
from bsk_rl.data import STINTaskReward
from bsk_rl.scene import CityTaskScenario
from bsk_rl.sats import ComputationSatellite
from bsk_rl.comm import LOSMultiCommunication
from bsk_rl.utils.orbital import walker_delta_args

# 运行训练循环
example_training_loop()


d:\miniconda\envs\Basilisk\Lib\site-packages\gymnasium\envs\registration.py:636: UserWarning: WARN: Overriding environment GeneralSatelliteTasking-v1 already in registry.
  logger.warn(f"Overriding environment {new_spec.id} already in registry.")
d:\miniconda\envs\Basilisk\Lib\site-packages\gymnasium\envs\registration.py:636: UserWarning: WARN: Overriding environment SatelliteTasking-v1 already in registry.
  logger.warn(f"Overriding environment {new_spec.id} already in registry.")
d:\miniconda\envs\Basilisk\Lib\site-packages\gymnasium\envs\registration.py:636: UserWarning: WARN: Overriding environment ConstellationTasking-v1 already in registry.
  logger.warn(f"Overriding environment {new_spec.id} already in registry.")
2025-12-17 14:58:59,939                                WARNING    Creating logger for new env on PID=46296. Old environments in process may now log times incorrectly.
2025-12-17 14:58:59,941 gym                            INFO       Resetting environment with seed=3282

已清除 52 个缓存模块


2025-12-17 14:59:00,344 sats.satellite.sat-0           INFO       <0.00> sat-0: Finding opportunity windows from 0.00 to 3600.00 seconds
2025-12-17 14:59:00,354 sats.satellite.sat-1           INFO       <0.00> sat-1: Finding opportunity windows from 0.00 to 3600.00 seconds
2025-12-17 14:59:00,372 sats.satellite.sat-2           INFO       <0.00> sat-2: Finding opportunity windows from 0.00 to 3600.00 seconds
2025-12-17 14:59:00,386 sats.satellite.sat-3           INFO       <0.00> sat-3: Finding opportunity windows from 0.00 to 3600.00 seconds
2025-12-17 14:59:00,403 sats.satellite.sat-4           INFO       <0.00> sat-4: Finding opportunity windows from 0.00 to 3600.00 seconds
2025-12-17 14:59:00,415 sats.satellite.sat-5           INFO       <0.00> sat-5: Finding opportunity windows from 0.00 to 3600.00 seconds
2025-12-17 14:59:00,447 gym                            INFO       <0.00> Environment reset
2025-12-17 14:59:00,450 gym                            INFO       <0.00> === STARTING S

环境已重置，智能体列表: ['sat-0', 'sat-1', 'sat-2', 'sat-3', 'sat-4', 'sat-5']
Step 1: Reward = -0.001, Cumulative = -0.001
Step 2: Reward = -0.001, Cumulative = -0.001
Step 3: Reward = -0.001, Cumulative = -0.002


2025-12-17 14:59:00,632 data.base                      INFO       <360.00> Total reward: {'sat-2': np.float64(-0.00027720000000006984), 'sat-4': np.float64(-0.0001870103239559423), 'sat-5': np.float64(-0.00027719999999998256)}
2025-12-17 14:59:00,633 comm.communication             INFO       <360.00> Communicating data in 4 directions.
2025-12-17 14:59:00,638 gym                            INFO       <360.00> Step reward: {'sat-2': np.float64(-0.00027720000000006984), 'sat-4': np.float64(-0.0001870103239559423), 'sat-5': np.float64(-0.00027719999999998256)}
2025-12-17 14:59:00,640 gym                            INFO       <360.00> === STARTING STEP ===
2025-12-17 14:59:00,664 sim.simulator                  INFO       <420.00> Max step duration reached
2025-12-17 14:59:00,665 data.base                      INFO       <420.00> Total reward: {'sat-2': np.float64(-0.00027720000000006984), 'sat-4': np.float64(-0.0001864872984286514), 'sat-5': np.float64(-0.00027719999999998256)}
2025-12-17 

...

Step 40: Reward = -0.001, Cumulative = -0.029


2025-12-17 14:59:01,813 gym                            INFO       <2760.00> === STARTING STEP ===
2025-12-17 14:59:01,865 sim.simulator                  INFO       <2820.00> Max step duration reached
2025-12-17 14:59:01,866 data.base                      INFO       <2820.00> Total reward: {'sat-2': np.float64(-0.00027719999999998256), 'sat-3': np.float64(-0.00027720000000006984), 'sat-4': np.float64(-0.0001844601777994234)}
2025-12-17 14:59:01,867 comm.communication             INFO       <2820.00> Communicating data in 4 directions.
2025-12-17 14:59:01,877 gym                            INFO       <2820.00> Step reward: {'sat-2': np.float64(-0.00027719999999998256), 'sat-3': np.float64(-0.00027720000000006984), 'sat-4': np.float64(-0.0001844601777994234)}
2025-12-17 14:59:01,881 gym                            INFO       <2820.00> === STARTING STEP ===
2025-12-17 14:59:01,904 sim.simulator                  INFO       <2880.00> Max step duration reached
2025-12-17 14:59:01,906 data.base

Step 50: Reward = -0.001, Cumulative = -0.036


2025-12-17 14:59:02,172 sim.simulator                  INFO       <3420.00> Max step duration reached
2025-12-17 14:59:02,173 data.base                      INFO       <3420.00> Total reward: {'sat-2': np.float64(-0.00027719999999998256), 'sat-3': np.float64(-0.00027720000000006984), 'sat-4': np.float64(-0.00018396645511762691)}
2025-12-17 14:59:02,174 comm.communication             INFO       <3420.00> Communicating data in 4 directions.
2025-12-17 14:59:02,179 gym                            INFO       <3420.00> Step reward: {'sat-2': np.float64(-0.00027719999999998256), 'sat-3': np.float64(-0.00027720000000006984), 'sat-4': np.float64(-0.00018396645511762691)}
2025-12-17 14:59:02,181 gym                            INFO       <3420.00> === STARTING STEP ===
2025-12-17 14:59:02,199 sim.simulator                  INFO       <3480.00> Max step duration reached
2025-12-17 14:59:02,200 data.base                      INFO       <3480.00> Total reward: {'sat-2': np.float64(-0.000277199999999

Step 60: Reward = -0.001, Cumulative = -0.044

Episode finished after 60 steps
Total reward: -0.044
Average reward per step: -0.001


## 参数说明

### ComputationDynModel 支持的参数

从 `@default_args` 装饰的 `setup_cpu_power_sink` 方法中定义：

| 参数名 | 默认值 | 单位 | 说明 |
|--------|--------|------|------|
| `cpuMinFrequency` | 0.1e9 | Hz | 最小 CPU 频率 |
| `cpuMaxFrequency` | 1.0e9 | Hz | 最大 CPU 频率 |
| `cpuWorkload` | 1000.0 | cycles/bit | 计算复杂度 |
| `cpuPowerDraw` | -20.0 | W | CPU 满载功耗（负值表示消耗） |
| `taskMinimumElevation` | 0.0 | rad | 任务可见性最小仰角 |

### 继承的参数（从父类）

从 `GroundStationDynModel` → `ImagingDynModel` → `BasicDynamicsModel` 继承：

- `batteryStorageCapacity`: 电池容量 [W·h]
- `instrumentBaudRate`: 仪器数据率 [bits]
- `transmitterBaudRate`: 下行速率 [baud]
- `mass`: 卫星质量 [kg]
- 等等...

**使用示例**：

```python
ComputationSatellite(
    name="sat-1",
    sat_args={
        "cpuMaxFrequency": 2.0e9,  # 自定义参数
        "batteryStorageCapacity": 80 * 3600,  # 继承参数
    }
)
```

## ✅ 参数传递流程验证

您的担心："如果放到了函数上面，那这些新参数如何配置？我不能通过 sat_args 配置了会找不到。"

**答案：完全可以配置！** 这正是 BSK-RL 框架的设计精妙之处。

### 🔄 完整的参数传递链路

```
用户代码                    框架内部处理
   ↓                           ↓
sat_args = {          →  1. collect_default_args(ComputationDynModel)
  "cpuMaxFrequency":         收集所有 @default_args 装饰的方法
   2.0e9                     包括 setup_cpu_power_sink.defaults
}                            
   ↓                           ↓
ComputationSatellite(    →  2. Satellite.default_sat_args(**sat_args)
  sat_args=sat_args           合并默认值 + 用户覆盖值
)                              
   ↓                           ↓
                          3. DynamicsModel.__init__(**kwargs)
                             将所有参数作为 kwargs 传入
   ↓                           ↓
                          4. _setup_dynamics_objects(**kwargs)
                             继续传递 kwargs
   ↓                           ↓
                          5. setup_cpu_power_sink(**kwargs)
                             从 kwargs 中提取参数：
                             - cpuMaxFrequency (用户值 2.0e9)
                             - cpuMinFrequency (默认值或用户值)
                             - ...
```

### 🔍 关键点

1. **`@default_args` 在 setup 方法上**：注册默认参数到 `setup_cpu_power_sink.defaults`
2. **`collect_default_args()`**：扫描类的所有方法，提取带 `.defaults` 的参数
3. **`**kwargs` 传递**：参数从 `__init__` → `_setup_dynamics_objects` → `setup_cpu_power_sink`
4. **函数签名提取**：`setup_cpu_power_sink(cpuMaxFrequency: float, ...)` 从 kwargs 中取出对应参数

In [ ]:
# ⚠️ 重要：需要强制重新加载模块以获取最新代码
import importlib
import sys

# 清除所有 bsk_rl 相关模块的缓存
modules_to_delete = [mod for mod in sys.modules.keys() if mod.startswith('bsk_rl')]
for mod_name in modules_to_delete:
    del sys.modules[mod_name]

print("已清除缓存的模块:", len(modules_to_delete), "个")

# 重新导入
from bsk_rl.sats import ComputationSatellite
from bsk_rl.utils.functional import collect_default_args

print("="*60)
print("✓ 模块已重新加载")
print("="*60)

# 调试：检查方法是否有 defaults 属性
method = getattr(ComputationSatellite.dyn_type, 'setup_cpu_power_sink')
print(f"\nsetup_cpu_power_sink.defaults = {method.defaults if hasattr(method, 'defaults') else 'NOT FOUND'}")

# 测试 1: 检查默认参数
default_args = ComputationSatellite.default_sat_args()
print("\n测试 1: 检查默认参数")
print("="*60)
print(f"cpuMinFrequency (默认): {default_args.get('cpuMinFrequency', 'NOT FOUND')}")
print(f"cpuMaxFrequency (默认): {default_args.get('cpuMaxFrequency', 'NOT FOUND')}")
print(f"cpuWorkload (默认): {default_args.get('cpuWorkload', 'NOT FOUND')}")
print(f"cpuPowerDraw (默认): {default_args.get('cpuPowerDraw', 'NOT FOUND')}")
print(f"taskMinimumElevation (默认): {default_args.get('taskMinimumElevation', 'NOT FOUND')}")

# 测试 2: 检查用户覆盖
print("\n" + "="*60)
print("测试 2: 用户自定义参数")
print("="*60)
custom_args = ComputationSatellite.default_sat_args(
    cpuMaxFrequency=5.0e9,  # 自定义为 5 GHz
    cpuMinFrequency=1.0e9,  # 自定义为 1 GHz
    batteryStorageCapacity=100 * 3600,  # 继承的参数也能覆盖
)
print(f"cpuMaxFrequency (自定义): {custom_args['cpuMaxFrequency']/1e9} GHz")
print(f"cpuMinFrequency (自定义): {custom_args['cpuMinFrequency']/1e9} GHz")
print(f"cpuWorkload (保持默认): {custom_args['cpuWorkload']}")
print(f"batteryStorageCapacity (自定义): {custom_args['batteryStorageCapacity']/3600} Wh")

# 测试 3: 验证在环境中生效
print("\n" + "="*60)
print("测试 3: 在实际卫星对象中验证参数")
print("="*60)
try:
    # 这就是用户实际的使用方式！
    sat = ComputationSatellite(
        name="test-sat",
        sat_args={
            "cpuMaxFrequency": 3.0e9,  # ← 用户通过这里配置
            "cpuMinFrequency": 0.8e9,
            "cpuWorkload": 300.0,
        }
    )
    
    print("✓ 卫星创建成功！")
    print(f"  sat_args_generator['cpuMaxFrequency'] = {sat.sat_args_generator['cpuMaxFrequency']/1e9} GHz")
    print(f"  sat_args_generator['cpuMinFrequency'] = {sat.sat_args_generator['cpuMinFrequency']/1e9} GHz")
    print(f"  sat_args_generator['cpuWorkload'] = {sat.sat_args_generator['cpuWorkload']} cycles/bit")
    
    print("\n" + "="*60)
    print("✅ 参数传递机制验证成功！")
    print("="*60)
    print("   用户可以正常通过 sat_args 配置 @default_args 装饰的参数。")
    
except Exception as e:
    print(f"✗ 创建失败: {e}")
    import traceback
    traceback.print_exc()


In [ ]:
## 🔍 终止条件与任务定义详解

### 1️⃣ `terminated` 和 `truncated` 的定义位置

这两个标志在 **[gym.py](../src/bsk_rl/gym.py)** 的 `GeneralSatelliteTasking` 类中定义：

```python
# gym.py 第 435-449 行
def _get_terminated(self) -> bool:
    """任务完成或所有卫星失效 → 正常结束"""
    if self.terminate_on_time_limit and self._get_truncated():
        return True
    else:
        return not all(
            satellite.is_alive() and not self.rewarder.is_terminated(satellite)
            for satellite in self.satellites
        )

def _get_truncated(self) -> bool:
    """超时或资源耗尽 → 提前截断"""
    return (self.simulator.sim_time >= self.time_limit) or any(
        self.rewarder.is_truncated(satellite) for satellite in self.satellites
    )
```

**关键区别**：
- **`terminated`**：任务目标完成，episode **正常结束**（正反馈）
- **`truncated`**：达到时间限制或资源耗尽，episode **提前截断**（中性/负反馈）

---

### 2️⃣ 回合最长时间设置

**已经设置好了！** 在 `create_stin_environment()` 中：

```python
env = ConstellationTasking(
    time_limit=3600.0,  # ← 1小时（3600秒）回合时长
    terminate_on_time_limit=False,  # 时间到时用 truncated（不用 terminated）
    ...
)
```

**参数说明**：
- `time_limit`：最大仿真时间（秒）
- `terminate_on_time_limit`：
  - `False` → 时间到用 `truncated=True`（推荐，表示"未完成"）
  - `True` → 时间到用 `terminated=True`（表示"任务完成"）

---

### 3️⃣ 任务定义在哪里？

**在 `STINTaskReward` 类中实现**（[stin_task_data.py](../src/bsk_rl/data/stin_task_data.py) 第 406-450 行）：

#### 📍 `is_truncated()` - 资源约束检查
```python
def is_truncated(self, satellite) -> bool:
    """检查是否应该截断 episode（资源耗尽）"""
    # 检查电池 SOC < 5% → 截断
    battery_soc = satellite.dynamics.powerMonitor.storageLevel / capacity
    if battery_soc < 0.05:
        logger.warning(f"{satellite.name} battery critically depleted")
        return True
    return False
```

#### 📍 `is_terminated()` - 任务完成检查
```python
def is_terminated(self, satellite) -> bool:
    """检查是否应该终止 episode（任务完成）"""
    # 当所有任务都完成或超时时终止
    total_tasks = len(self.scenario.tasks)
    completed = sum(sat.completed_tasks_count for sat in satellites)
    expired = sum(sat.expired_tasks_count for sat in satellites)
    
    if completed + expired >= total_tasks:
        return True  # 所有任务处理完毕
    return False
```

---

### 4️⃣ 如何自定义终止条件？

**方法 1：修改 `STINTaskReward` 类**

编辑 [stin_task_data.py](../src/bsk_rl/data/stin_task_data.py) 中的 `is_terminated()` 和 `is_truncated()` 方法。

**方法 2：创建自定义 Rewarder 子类**

```python
from bsk_rl.data import STINTaskReward

class CustomSTINReward(STINTaskReward):
    def is_terminated(self, satellite) -> bool:
        # 自定义终止条件：例如完成率达到 80%
        if self.scenario.get_completion_rate() >= 0.8:
            return True
        return super().is_terminated(satellite)
    
    def is_truncated(self, satellite) -> bool:
        # 自定义截断条件：例如平均队列长度过大
        avg_queue = np.mean([len(s.task_queue) for s in self.satellites])
        if avg_queue > 50:
            logger.warning("Task queue overflow, truncating")
            return True
        return super().is_truncated(satellite)
```

**方法 3：在环境配置中指定**

```python
env = ConstellationTasking(
    rewarder=CustomSTINReward(...),
    time_limit=3600.0,  # 1 小时强制截断
    terminate_on_time_limit=False,
    ...
)
```

---

### 5️⃣ 任务场景定义

**在 `CityTaskScenario` 中定义**（[stin_scenario.py](../src/bsk_rl/scene/stin_scenario.py)）：

```python
task_scenario = CityTaskScenario(
    n_tasks=20,                    # 总任务数
    n_select_from=100,             # 从前 100 大城市中选择
    data_size_range=(1e6, 10e6),   # 数据大小范围 [Mb]
    workload_range=(100, 1000),    # 计算复杂度范围 [cycles/bit]
    max_delay_range=(5.0, 20.0),   # 最大时延约束范围 [s]
    task_arrival_rate=0.1,         # 任务到达率 [tasks/s]
)
```

**关键属性**：
- `n_tasks`：回合内总任务数（完成后 `terminated=True`）
- `max_delay_range`：每个任务的时延约束（超时算失败）
- `task_arrival_rate`：任务生成速率

---

### 📊 总结

| 概念 | 定义位置 | 含义 | 当前配置 |
|------|---------|------|---------|
| `terminated` | `gym.py` + `STINTaskReward.is_terminated()` | 任务完成正常结束 | 所有任务完成/超时 |
| `truncated` | `gym.py` + `STINTaskReward.is_truncated()` | 资源耗尽提前截断 | 电池 < 5% 或时间 > 3600s |
| `time_limit` | `ConstellationTasking(...)` | 回合最大时长 | **3600 秒（1小时）** |
| 任务定义 | `CityTaskScenario(...)` | 任务属性和约束 | 20 个任务，5-20s 时延 |

**推荐配置**：
- ✅ `time_limit=3600.0`（已设置）
- ✅ `terminate_on_time_limit=False`（时间到用 truncated）
- ✅ 在 `STINTaskReward` 中实现自定义终止逻辑
